# Calibração conforme (S5.7) — 3 grupos em um notebook

**Autor**: Massanori  
**Data**: 19/05/2026  
**Descrição**: Calibra os 3 modulos treinados (ResM, QR, QR-Lesion) sobre o split `cal` e avalia cobertura empírica no split `test`. Implementa Romano, Patterson & Candès (2019) para QR e Edupuganti et al. (2021) para ResM. Pixel-wise, ~75M amostras (Angelopoulos et al., 2022).

## Pré-requisitos (5 inputs)

- Settings → Accelerator = **T4 x1** (ou P100 — leve, mas precisa GPU)
- Add Input (5 datasets):
  1. `tcc-mri-recons-varnet-brain-4x` (S4)
  2. `tcc-mri-lesion-masks` (S3)
  3. `tcc-mri-resm-checkpoints` (S5.2 — Grupo A)
  4. `tcc-mri-qr-checkpoints` (S5.3 — Grupo B)
  5. `tcc-mri-qr-lesion-checkpoints` (S5.4 — Grupo C)

## Fluxo

1. Célula 2: Setup
2. Célula 3: Diagnóstico (5 datasets + GPU)
3. Célula 4: DataLoaders (cal + test, com masks)
4. Célula 5: Loop sobre os 3 grupos (calibrate + evaluate)
5. Célula 6: Tabela comparativa + salva resultados
6. Célula 7: (opcional) publica como Kaggle dataset `tcc-mri-calibration-results`

## Tempo esperado

Total: ~15-30 min. Cada grupo: ~3-5 min cal + ~3-5 min test em T4.

In [ ]:
# Célula 2 — Setup
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/KR0N0S7/tcc-mri-uncertainty.git'
REPO_DIR = Path('/kaggle/working/tcc-mri-uncertainty')

if REPO_DIR.is_dir():
    print(f'Repo ja existe; git pull...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print('\nHEAD:')
subprocess.run(['git', '-C', str(REPO_DIR), 'log', '-1', '--oneline'], check=False)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    '-r', str(REPO_DIR / 'requirements.txt'),
    'python-dotenv',
], check=True)
print('Setup OK')

In [ ]:
# Célula 3 — Diagnóstico: 5 datasets + GPU
import os
import subprocess
import sys
from pathlib import Path

SLUGS = {
    'recons': 'tcc-mri-recons-varnet-brain-4x',
    'masks': 'tcc-mri-lesion-masks',
    'resm':  'tcc-mri-resm-checkpoints',
    'qr':    'tcc-mri-qr-checkpoints',
    'qrl':   'tcc-mri-qr-lesion-checkpoints',
}


def kaggle_input_candidates(slug):
    cs = [Path('/kaggle/input') / slug]
    dr = Path('/kaggle/input/datasets')
    if dr.is_dir():
        for u in dr.iterdir():
            if u.is_dir():
                cs.append(u / slug)
    return cs


def find_path(slug, want_pattern, min_count=1):
    """Encontra dataset que tem >= min_count arquivos casando want_pattern."""
    for cand in kaggle_input_candidates(slug):
        if not cand.is_dir():
            continue
        # Padrao 1: arquivos no proprio diretorio
        if len(list(cand.glob(want_pattern))) >= min_count:
            return cand
        if any((cand / s).is_dir() for s in ('train', 'val', 'cal', 'test')):
            return cand  # recons-like
        # Padrao 2: aninhamento de 1 nivel
        for sub in cand.iterdir():
            if sub.is_dir():
                if len(list(sub.glob(want_pattern))) >= min_count:
                    return sub
                if any((sub / s).is_dir() for s in ('train', 'val', 'cal', 'test')):
                    return sub
    raise FileNotFoundError(f'{slug} nao encontrado.')


print('=' * 60)
print('DIAGNOSTICO')
print('=' * 60)

print('\n[1] Conteudo de /kaggle/input/:')
subprocess.run(['ls', '-la', '/kaggle/input/'], check=False)
if Path('/kaggle/input/datasets').is_dir():
    subprocess.run(['ls', '-la', '/kaggle/input/datasets/'], check=False)

# Resolve os 5 paths
RECONS_ROOT = find_path(SLUGS['recons'], '*.npz', min_count=10)
MASKS_ROOT = find_path(SLUGS['masks'], '*.pt', min_count=100)
RESM_CKPT = find_path(SLUGS['resm'], 'best.pt', min_count=1) / 'best.pt'
QR_CKPT = find_path(SLUGS['qr'], 'best.pt', min_count=1) / 'best.pt'
QRL_CKPT = find_path(SLUGS['qrl'], 'best.pt', min_count=1) / 'best.pt'

print(f'\n[2] Paths resolvidos:')
print(f'  RECONS_ROOT: {RECONS_ROOT}')
print(f'  MASKS_ROOT:  {MASKS_ROOT} ({len(list(MASKS_ROOT.glob("*.pt")))} .pt)')
print(f'  RESM_CKPT:   {RESM_CKPT} ({RESM_CKPT.stat().st_size / 1e6:.1f} MB)')
print(f'  QR_CKPT:     {QR_CKPT} ({QR_CKPT.stat().st_size / 1e6:.1f} MB)')
print(f'  QRL_CKPT:    {QRL_CKPT} ({QRL_CKPT.stat().st_size / 1e6:.1f} MB)')

import torch
print(f'\n[3] CUDA: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    raise RuntimeError('GPU nao habilitada. Settings -> Accelerator -> T4 x1.')
print(f'  GPU: {torch.cuda.get_device_name(0)}')

print('\n' + '=' * 60)
print('DIAGNOSTICO OK')
print('=' * 60)

In [ ]:
# Célula 4 — DataLoaders para cal e test (com masks pro Coverage_lesion)
from torch.utils.data import DataLoader
from src.data import ReconsSliceDataset

cal_ds = ReconsSliceDataset(RECONS_ROOT / 'cal', masks_dir=MASKS_ROOT)
test_ds = ReconsSliceDataset(RECONS_ROOT / 'test', masks_dir=MASKS_ROOT)

cal_loader = DataLoader(
    cal_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True,
)
test_loader = DataLoader(
    test_ds, batch_size=1, shuffle=False, num_workers=2, pin_memory=True,
)
print(f'cal:  {len(cal_ds)} slices')
print(f'test: {len(test_ds)} slices')

In [ ]:
# Célula 5 — Loop sobre os 3 grupos: calibrate + evaluate.
#
# A) Grupo A (ResM): calibracao multiplicativa, |x-y|/u, intervalo [x-qhat*u, x+qhat*u]
# B) Grupo B (QR):  calibracao aditiva,        max(l-y, y-u),  intervalo [l-qhat, u+qhat]
# C) Grupo C (QR-L): mesma calibracao que B (arquitetura identica via alias)
import time

import torch

from src.calibration import calibrate, evaluate
from src.models import (
    QuantileRegressionLesionModule,
    QuantileRegressionModule,
    ResidualMagnitudeModule,
)
from src.training import load_checkpoint

ALPHA = 0.10
DEVICE = 'cuda'

groups = [
    {'name': 'A', 'module_cls': ResidualMagnitudeModule,
     'checkpoint': RESM_CKPT, 'kind': 'resm'},
    {'name': 'B', 'module_cls': QuantileRegressionModule,
     'checkpoint': QR_CKPT, 'kind': 'qr'},
    {'name': 'C', 'module_cls': QuantileRegressionLesionModule,
     'checkpoint': QRL_CKPT, 'kind': 'qr'},
]

results = {}

for g in groups:
    print(f'\n{"=" * 60}')
    print(f'Grupo {g["name"]} (kind={g["kind"]})')
    print(f'  checkpoint: {g["checkpoint"].name}')
    print('=' * 60)

    t0 = time.time()
    module = g['module_cls'](chans=32, num_pool_layers=4)
    load_checkpoint(g['checkpoint'], module, device=DEVICE)
    module = module.to(DEVICE).eval()
    print(f'  modulo carregado em {time.time()-t0:.1f}s')

    print(f'  Calibrando no split cal ({len(cal_ds)} slices)...')
    t0 = time.time()
    qhat, _scores = calibrate(
        module, cal_loader, alpha=ALPHA, kind=g['kind'], device=DEVICE,
    )
    print(f'  qhat = {qhat:.6f}  ({time.time()-t0:.1f}s)')

    print(f'  Avaliando no split test ({len(test_ds)} slices)...')
    t0 = time.time()
    metrics = evaluate(
        module, test_loader, qhat, kind=g['kind'], device=DEVICE,
    )
    print(f'  done em {time.time()-t0:.1f}s')
    print(f'  coverage_global: {metrics["coverage_global"]:.4f}  (nominal {1-ALPHA:.2f})')
    print(f'  coverage_lesion: {metrics["coverage_lesion"]:.4f}')
    print(f'  mean_width:      {metrics["mean_width"]:.6f}')
    print(f'  n_lesion pixels: {metrics["n_lesion"]:,}')

    results[g['name']] = metrics

    # Libera GPU memory antes do proximo modulo
    del module
    torch.cuda.empty_cache()

print('\nTodos os 3 grupos calibrados e avaliados.')

In [ ]:
# Célula 6 — Tabela comparativa + persistencia em JSON.
#
# Esta tabela e o output principal do S5.7. Ela alimenta a discussao
# do TCC: se cobertura global ~ 1-alpha em todos os 3, validacao OK;
# se coverage_lesion(C) > coverage_lesion(B) >= coverage_lesion(A),
# a contribuicao original do TCC esta empiricamente suportada.
import json
from pathlib import Path

alpha = 0.10

print(f'\nCalibration Results (alpha={alpha}, cobertura nominal {1-alpha:.2f})')
print('=' * 78)
print(f'{"Grupo":<6} {"qhat":>10}  {"Cov_global":>11}  {"Cov_lesion":>11}  {"Mean_width":>12}')
print('-' * 78)
for name in ('A', 'B', 'C'):
    r = results[name]
    print(
        f'{name:<6} '
        f'{r["qhat"]:>10.6f}  '
        f'{r["coverage_global"]:>11.4f}  '
        f'{r["coverage_lesion"]:>11.4f}  '
        f'{r["mean_width"]:>12.6f}'
    )
print('=' * 78)

# Sanity checks (logging)
print('\nSanity checks:')
for name in ('A', 'B', 'C'):
    r = results[name]
    gap = abs(r['coverage_global'] - (1 - alpha))
    status = '✓' if gap < 0.02 else '⚠' if gap < 0.05 else '✗'
    print(f'  {status} {name}: |coverage_global - {1-alpha:.2f}| = {gap:.4f}')

# Persistencia: salva resultados em JSON para uso na proxima sessao (S5.8)
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-calibration-results')
OUTPUT_DIR.mkdir(exist_ok=True)

# Trim per_slice arrays para JSON (mantem como float, util para Friedman test)
summary = {}
for name, r in results.items():
    summary[name] = {
        'qhat': r['qhat'],
        'kind': r['kind'],
        'coverage_global': r['coverage_global'],
        'coverage_lesion': r['coverage_lesion'],
        'mean_width': r['mean_width'],
        'n_total': r['n_total'],
        'n_lesion': r['n_lesion'],
        'per_slice_coverage': r['per_slice_coverage'],
        'per_slice_width': r['per_slice_width'],
    }
summary['_meta'] = {
    'alpha': alpha,
    'n_cal_slices': len(cal_ds),
    'n_test_slices': len(test_ds),
}

out_file = OUTPUT_DIR / 'calibration_summary.json'
out_file.write_text(json.dumps(summary, indent=2), encoding='utf-8')
print(f'\nResumo salvo em {out_file} ({out_file.stat().st_size / 1024:.1f} kB)')

In [ ]:
# Célula 7 — (Opcional) publica resultados como Kaggle dataset para o S5.8.
import json
import subprocess
from pathlib import Path

KAGGLE_USER = 'massanorikishi'
DATASET_SLUG = 'tcc-mri-calibration-results'
DATASET_TITLE = 'TCC MRI Calibration Results (S5.7)'
OUTPUT_DIR = Path('/kaggle/working/tcc-mri-calibration-results')

metadata = {
    'title': DATASET_TITLE,
    'id': f'{KAGGLE_USER}/{DATASET_SLUG}',
    'licenses': [{'name': 'CC0-1.0'}],
}
(OUTPUT_DIR / 'dataset-metadata.json').write_text(
    json.dumps(metadata, indent=2), encoding='utf-8'
)

print('Tentando criar dataset...')
result = subprocess.run(
    ['kaggle', 'datasets', 'create', '-p', str(OUTPUT_DIR), '-r', 'zip'],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    result = subprocess.run(
        ['kaggle', 'datasets', 'version', '-p', str(OUTPUT_DIR),
         '-m', 'Update from S5.7 calibration NB', '-r', 'zip'],
        capture_output=True, text=True,
    )
    print(result.stdout); print(result.stderr)

print(f'\nhttps://www.kaggle.com/datasets/{KAGGLE_USER}/{DATASET_SLUG}')

## Próximos passos (S5.8 — métricas estatísticas e análise)

O dataset `tcc-mri-calibration-results` contém, para cada grupo:

- `qhat`, `coverage_global`, `coverage_lesion`, `mean_width` (escalares)
- `per_slice_coverage`, `per_slice_width` (arrays para Friedman test)

### `notebooks/analysis_S5_8.ipynb` (estatística)

Vai computar:

1. **Friedman test** sobre `per_slice_coverage` (A vs B vs C) — detecta diferenças gerais
2. **Nemenyi post-hoc** — identifica quais pares de grupos diferem (Demšar, 2006)
3. **Holm-Bonferroni correction** para múltiplas comparações (Holm, 1979)
4. **BCa bootstrap** para CI 95% em subgrupos pequenos (Efron, 1987)
5. **Clopper-Pearson interval** para Coverage_lesion (binomial exato)
6. **IoU_uncertainty** — alinhamento espacial entre alta uncertainty e alto erro
7. **ULAS** — métrica original (gradient alignment dentro de lesões)

Refs: Friedman (1937); Demsar (2006); Holm (1979); Efron (1987); Clopper-Pearson (1934).